# Semana 4: Dirty Data Challenge: el censo de 1994

**Introducción a la Ciencia de Datos** · Dudbil Olvasada Pabón Riaño · UNAB

Cuaderno guiado. Cada celda corresponde a un ejercicio de la guía de laboratorio; los resultados deben coincidir con sus puntos de control. Los datos se leen desde el repositorio del curso.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
BASE = "https://raw.githubusercontent.com/dudbilopr/Introduccion-a-la-ciencia-de-datos/main/Semanas/Datos/"

Roles: **forense** (diagnostica), **juez** (decide y justifica) y **cronista** (escribe la bitácora). Datos: Adult, censo de Estados Unidos de 1994, UCI, CC BY 4.0.

## 1. Inventario: lo que isna() no ve

In [ ]:
df = pd.read_csv(BASE + "censo_adult.csv")
print(df.shape, " vacíos según pandas:", df.isna().sum().sum())
signos = (df == "?").sum()
print(signos[signos > 0])
alguna = (df == "?").any(axis=1)
print("Filas con algún '?':", alguna.sum(), "=", round(alguna.mean() * 100, 1), "%")
print("Duplicadas:", df.duplicated().sum())
print("capital_gain == 99999:", (df["capital_gain"] == 99999).sum(), " edad 90:", (df["age"] == 90).sum(), " edad 89:", (df["age"] == 89).sum())

## 2. ¿Faltan al azar? Patrones y edades

In [ ]:
print(pd.crosstab(df["workclass"] == "?", df["occupation"] == "?"))
print(df[(df["occupation"] == "?") & (df["workclass"] != "?")]["workclass"].value_counts())
sin = df[df["workclass"] == "?"]["age"]; con = df[df["workclass"] != "?"]["age"]
print("Menores de 25 (%): sin tipo de empleo", round((sin < 25).mean() * 100, 1), " con tipo", round((con < 25).mean() * 100, 1))
print("60 o más (%): sin tipo de empleo", round((sin >= 60).mean() * 100, 1), " con tipo", round((con >= 60).mean() * 100, 1))

**Pregunta del juez:** ¿son MCAR, MAR o MNAR? Escriban la evidencia.

## 3. Decisión 1: borrar filas

In [ ]:
alto = df["income"] == ">50K"
print("Ingreso alto, todos:", round(alto.mean() * 100, 1), "%")
print("Ingreso alto, filas con '?':", round(alto[alguna].mean() * 100, 1), "%")
limpio = df[~alguna]
print("Tras borrar:", len(limpio), "filas; ingreso alto", round((limpio["income"] == ">50K").mean() * 100, 1), "%")

## 4. Decisión 2: imputar con la moda

In [ ]:
moda_wc = df.loc[df["workclass"] != "?", "workclass"].mode()[0]
moda_oc = df.loc[df["occupation"] != "?", "occupation"].mode()[0]
print("Modas:", moda_wc, moda_oc)
antes = (df["workclass"] == moda_wc).sum() / (df["workclass"] != "?").sum()
imp = df["workclass"].replace("?", moda_wc)
print("Private:", round(antes * 100, 1), "% ->", round((imp == moda_wc).mean() * 100, 1), "%")
oc_imp = df["occupation"].replace("?", moda_oc)
print("Ingreso alto en", moda_oc, ":", round(alto[df["occupation"] == moda_oc].mean() * 100, 1), "% ->", round(alto[oc_imp == moda_oc].mean() * 100, 1), "%")

## 5. Decisión 3: marcar y conservar

In [ ]:
marcado = df.copy()
for c in ["workclass", "occupation", "native_country"]:
    marcado[c + "_falta"] = (marcado[c] == "?").astype(int)
    marcado[c] = marcado[c].replace("?", "Desconocido")
marcado.loc[marcado["workclass"] == "Never-worked", "occupation"] = "No aplica"
print(marcado["occupation"].value_counts().tail(4))
print("Filas conservadas:", len(marcado))

## 6. Atípicos: tres reglas para las horas

In [ ]:
h = df["hours_per_week"]
print("Exactamente 40 horas:", round((h == 40).mean() * 100, 1), "%")
z = (h - h.mean()) / h.std()
q1, q3 = h.quantile(.25), h.quantile(.75)
from scipy import stats
mad = stats.median_abs_deviation(h)
print("Media ± 3 DE:", (z.abs() > 3).sum())
print("RIC: límites", q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1), "->", ((h < q1 - 1.5 * (q3 - q1)) | (h > q3 + 1.5 * (q3 - q1))).sum())
print("Mediana ± 3 MAD (MAD =", mad, "):", ((h - h.median()).abs() / (1.4826 * mad) > 3).sum())
h.plot.hist(bins=50); plt.title("El pico de 40 horas engaña a las reglas automáticas"); plt.xlabel("Horas por semana"); plt.show()

## 7. Topes de registro

In [ ]:
g = df["capital_gain"]
print("Ceros:", round((g == 0).mean() * 100, 1), "%  máximo sin tope:", g[g != 99999].max())
print("Media con tope:", round(g.mean()), " sin tope:", round(g[g != 99999].mean()))
print(df["age"].value_counts().sort_index().tail(6))

## 8. Bitácora del equipo
Escriban aquí, en una celda de texto, cada decisión: qué hicieron, por qué y con qué efecto (con números). Si usaron un asistente de IA para proponer opciones, declárenlo: herramienta, prompt y qué verificaron.

## Declaración de uso de IA

Si usó una herramienta de IA, escriba aquí: cuál, para qué, el prompt, la versión del modelo y qué verificó usted.